# Python Patterns You Will See in Later Labs

This notebook extends the first Python primer. The goal is to help you read the helper patterns that appear often in Labs 2-5.

It assumes you already:

- completed steps 1-4 in `lab0_01_llm_foundations/01_instructions.md`
- ran `03_python_basics_notebook.ipynb`

The focus here is still notebook reading, not advanced programming.

Tip for non-CS students: comments begin with `#`. When a line feels compact, read the comment first and then the code line under it.

In [ ]:
# Import the same file-reading helpers used in later labs.
import csv
import json
# `sys` lets the notebook adjust Python's import path.
import sys
from pathlib import Path

LAB_NAME = "lab0_00_python_basics"
# `Path.cwd()` means the folder where this notebook is running.
lab_dir = Path.cwd()
if lab_dir.name != LAB_NAME:
    raise RuntimeError(
        f"Open this notebook from the {LAB_NAME} folder after completing steps 1-4 in lab0_01_llm_foundations."
    )

# The repo root is one folder above this lab folder.
repo_root = lab_dir.parent
src_dir = repo_root / "src"
# Later labs import code from `src/`, so we add that folder to Python's search path.
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

data_dir = lab_dir / "data"
print("Notebook folder:", lab_dir.name)
print("Repo root:", repo_root.name)
print("Data files:", sorted(path.name for path in data_dir.iterdir()))


## 1. `with` and File Reading

Later labs often use `with ... open(...) as handle:` when they load JSON or CSV files.

Read it like this:

- open the file for a short block of work
- use the file inside the indented block
- let Python close the file automatically afterward


In [ ]:
# Build two Path objects that point to the small practice files.
case_packet_path = data_dir / "case_packet.json"
timeline_path = data_dir / "timeline.csv"

# Open the JSON file and turn it into Python data.
with case_packet_path.open("r", encoding="utf-8") as handle:
    case_packet = json.load(handle)

# Open the CSV file and turn each row into a dictionary.
with timeline_path.open("r", encoding="utf-8", newline="") as handle:
    timeline_rows = list(csv.DictReader(handle))

print("Case ID:", case_packet["case_id"])
print("First note:", case_packet["notes"][0])
print("Row count:", len(timeline_rows))
print("First event:", timeline_rows[0]["event"])


## 2. `sorted(...)` and List Comprehensions

You will often see notebooks use `sorted(...)` so every student sees the same output order.

A list comprehension is a compact way to build a new list. Read it as:

- take one item at a time
- pull out the part you want
- store the results in a new list


In [ ]:
# Read this list comprehension as:
# "for each row, take the event field and build a new list from those values."
unsorted_events = [row["event"] for row in timeline_rows]
# `sorted(...)` returns a new sorted version instead of changing the old list in place.
sorted_events = sorted(unsorted_events)
# This comprehension builds another new list from the note values.
upper_notes = [note.upper() for note in case_packet["notes"]]

print("Unsorted events:", unsorted_events)
print("Sorted events:", sorted_events)
print("Upper-case notes:", upper_notes)


## 3. Dictionary Comprehensions and Lookup Tables

Later labs often build a lookup dictionary from rows that were loaded from evidence files.

Read this pattern as:

- use one field as the key
- store the whole row, or one value from the row, under that key
- make future lookups easier and faster


In [ ]:
# Build a lookup dictionary where each event name points to the whole row.
rows_by_event = {row["event"]: row for row in timeline_rows}
# Build a second lookup dictionary where each event name points only to its source.
source_by_event = {row["event"]: row["source"] for row in timeline_rows}

print("Lookup keys:", sorted(rows_by_event))
print("network_upload row:", rows_by_event["network_upload"])
print("message_sent source:", source_by_event["message_sent"])


## 4. `enumerate(...)` and Type Hints

`enumerate(...)` is a helpful reading pattern when the notebook wants both:

- a counter like `1, 2, 3`
- the item itself

You will also see type hints such as `path: Path` or `-> list[dict]`. In this course, treat them as reading aids that suggest what the function expects and what it returns.


In [ ]:
# Type hints help the reader understand the intended input and output shapes.
def read_timeline_rows(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))


def summarize_events(rows: list[dict]) -> dict:
    # Return a new dictionary summary built from the input rows.
    return {
        "row_count": len(rows),
        "events": [row["event"] for row in rows],
    }


fresh_rows = read_timeline_rows(timeline_path)
summary = summarize_events(fresh_rows)

# `enumerate(..., start=1)` gives both the row number and the row itself.
for index, row in enumerate(fresh_rows, start=1):
    print(f"{index}. {row['timestamp']} -> {row['event']}")

print("Summary:", summary)


## 5. Imports From Course Code and `@tool`

Labs 2-5 import helpers from the course code in `src/agentic_patterns/...`.

You do not need decorator theory for this course. It is enough to read `@tool` as:

- take the function below it
- wrap it so an agent can treat it like a tool
- give it a name, signature, and `.run(...)` method


In [ ]:
# Import the `tool` helper from the course code.
from agentic_patterns.tool_pattern.tool import tool


# Read `@tool` as: turn the next function into a course tool object.
@tool
def get_case_status(case_id: str) -> str:
    """Return a short practice status string for one case."""
    return f"{case_id}: practice review pending"


# `fn_signature` is stored as JSON text, so we load it into a Python dictionary for display.
tool_signature = json.loads(get_case_status.fn_signature)
print("Tool name:", get_case_status.name)
print("Tool signature:", tool_signature)
# `.run(...)` executes the wrapped function.
print("Tool run result:", get_case_status.run(case_id=case_packet["case_id"]))


## 6. Mini Later-Lab Style Example

This last cell combines several patterns you will see later:

- `Path` objects
- `with` for file reading
- `sorted(...)`
- list and dictionary comprehensions
- a small helper tool result

The goal is not to memorize the syntax. The goal is to practice slowing down and reading one piece at a time.


In [ ]:
# This generator expression walks through the files in the data folder and keeps their names.
artifact_files = sorted(path.name for path in data_dir.iterdir() if path.is_file())
# This builds a sorted list of event names from the row dictionaries.
events = sorted(row["event"] for row in timeline_rows)
# This lookup dictionary makes it easy to jump straight to a row by its source.
rows_by_source = {row["source"]: row for row in timeline_rows}

# Pull the results together into one summary dictionary, like later labs often do.
mini_case_view = {
    "case_id": case_packet["case_id"],
    "artifact_files": artifact_files,
    "events": events,
    "upload_detail": rows_by_source["network"]["detail"],
    "status_tool_result": get_case_status.run(case_id=case_packet["case_id"]),
}

mini_case_view


If you can read this notebook comfortably, you are ready for the Python reading patterns that appear most often in Labs 2-5.